# M6b · Build the three marts views

**Goal:** (re)create `marts.v_device_risk_profile`, `marts.v_ml_features_driver_behavior`, `marts.v_fleet_risk_dashboard`.

Views are CREATE-OR-REPLACE: always idempotent.

**Exit criterion:** each view returns rows.


In [1]:
# === Setup ===
from __future__ import annotations
import sys, pathlib
# Make the src/ package importable even if pip install -e was skipped.
PROJECT_ROOT = pathlib.Path().resolve().parents[1] if pathlib.Path().resolve().name != 'accent-fleet-analytics' else pathlib.Path().resolve()
for candidate in (PROJECT_ROOT, PROJECT_ROOT.parent):
    src = candidate / 'src'
    if src.exists() and str(src) not in sys.path:
        sys.path.insert(0, str(src))
        break

from accent_fleet.config import settings
from accent_fleet.db import get_engine, transaction
from sqlalchemy import text

s = settings()
print(f"DB = {s.pg_user}@{s.pg_host}:{s.pg_port}/{s.pg_database}")
print(f"Schemas: staging={s.pg_schema_staging}, warehouse={s.pg_schema_warehouse}, marts={s.pg_schema_marts}")


DB = medali_dev@localhost:15432/accent_data
Schemas: staging=staging, warehouse=warehouse, marts=marts


## 2. Inputs

In [2]:
VIEW_SQL_FILES = [
    '21_v_device_risk_profile.sql',
    '22_v_ml_features.sql',
    '23_v_fleet_risk_dashboard.sql',
]


## 3. Execute

In [3]:
from accent_fleet.db import run_sql_file
with transaction() as conn:
    for f in VIEW_SQL_FILES:
        run_sql_file(conn, f)
        print('applied', f)


applied 21_v_device_risk_profile.sql
applied 22_v_ml_features.sql
applied 23_v_fleet_risk_dashboard.sql


## 4. Inspect

In [4]:
import pandas as pd
with get_engine().connect() as conn:
    for v in ['v_device_risk_profile','v_ml_features_driver_behavior','v_fleet_risk_dashboard']:
        n = conn.execute(text(f'SELECT COUNT(*) FROM marts.{v}')).scalar_one()
        print(f'marts.{v}: {n:,} rows')
    display(pd.read_sql(text('SELECT * FROM marts.v_device_risk_profile ORDER BY risk_score DESC LIMIT 10'), conn))


marts.v_device_risk_profile: 448 rows
marts.v_ml_features_driver_behavior: 23,642 rows
marts.v_fleet_risk_dashboard: 4 rows


,tenant_id,device_id,latest_month,trips_3m,distance_3m,overspeed_3m,severe_overspeed_3m,alerts_3m,risk_score,risk_category
0,235,425271,2026-04,662,6163.663611,359,198,0,45,moderate
1,264,436322,2026-04,232,8176.614913,787,0,0,43,moderate
2,264,436369,2026-04,1020,24834.243059,2811,0,0,41,moderate
3,264,436325,2026-04,352,22246.571582,3088,0,0,40,moderate
4,235,425218,2026-01,541,10491.792708,1095,0,0,40,moderate
5,264,436308,2026-04,369,22257.264562,3088,0,0,39,moderate
6,264,436366,2026-04,928,27329.432822,2796,0,0,38,moderate
7,235,425310,2026-04,756,19450.819034,3438,0,0,38,moderate
8,235,425243,2026-04,420,5601.724893,496,0,0,38,moderate
9,264,436306,2026-04,562,8425.608931,1153,0,0,38,moderate
